# Exercise 5-1-1: Detecting Non-Answers in Earnings Conference Calls

This exercise replicates the core of the case study in Section 4 of de Kok ([2025](https://doi.org/10.1287/mnsc.2023.03253)) — *"ChatGPT for Textual Analysis? How to Use Generative LLMs in Accounting Research," Management Science* 71(9), 7888–7906 — on the 13 Tesla Q4 earnings call transcripts in [data/conference_call_transcript/](../data/conference_call_transcript/).

**The research task.** In the Q&A section of an earnings call, analysts ask questions and managers answer them — except when they don't. A **non-answer** is a response in which the manager signals an inability or an unwillingness to provide (part of) the information that was asked for: *"we don't disclose that", "it's too early to say", "we don't give guidance on that."* Hollander, Pronk and Roelofsen (2010, *JAR* 48(3):531–563) show these silences are informative, but they had to hand-code them, which took months. Gow, Larcker and Zakolyukina (2021, *JAR* 59(4):1349–1384) automate the coding with regular expressions; in de Kok's replication that measure reaches 86% accuracy but only a 0.49 F1 score on the non-answer class, missing 57% of the true non-answers. They tried and failed to do better with traditional machine learning, calling it *"not as straightforward as it seems"* and leaving *"this challenge for future research."*

That is the challenge de Kok (2025) takes up. Deciding whether a response is a non-answer requires reading the answer **in the context of the question**, tolerating genuine ambiguity, and coping with messy transcripts — precisely the kind of judgement task that used to require a research assistant, and precisely where a generative LLM earns its cost. A zero-shot ChatGPT prompt alone gets him to 91% accuracy (0.72 F1); the full pipeline reaches 96% (0.87 F1), a 70% reduction in the error rate relative to Gow et al. (2021).

**What we build here** is the zero-shot column of his Table 1 (column (3)): raw transcript in → one row per Q&A pair → a 0/1 non-answer label out, plus an evaluation of whether that label can be trusted.

de Kok's four-step framework (§3 of the paper) maps onto this notebook as follows:

| Framework step | Where in this notebook |
| --- | --- |
| 1. Define and understand your problem | Step 1 |
| 2. Decide on the approach and model | Step 3 |
| 3. Develop your prompt | Step 4 |
| 4. Evaluate the construct validity | Step 6 |


## Step 0. Setup

In [ ]:
import json
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

import os

load_dotenv()

In [ ]:
# Notebooks in this repo are run either from the module folder or from the repo
# root, so locate the repo root by looking for the shared `data/` folder.
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
CALL_DIR = ROOT / "data" / "conference_call_transcript"
OUT_DIR = ROOT / "data" / "nonanswer"      # everything we generate goes here
OUT_DIR.mkdir(parents=True, exist_ok=True)

CALL_FILES = sorted(CALL_DIR.glob("*.txt"))
[f.name for f in CALL_FILES]

In [ ]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5.6"

## Step 1. Define and understand the problem

> *"Any model, irrespective of how powerful it is, will not perform well if it lacks the necessary information to make a prediction or classification."* — de Kok (2025, §3.1)

Before writing a single prompt, be explicit about **what you are measuring** and **what information the model needs** to measure it.

**The construct.** A response is a non-answer if the manager indicates an inability or an unwillingness to provide (part of) the information the question asked for. Following the dimensions in de Kok's Appendix A, that covers:

* *Cannot give* — declining for proprietary, competitive, policy or legal reasons.
* *Do not know* — the information is not available, or it is too early to tell.
* And it shows up as a **complete refusal**, a **qualitative deflection** (an answer, but not to the question asked), or a **range/percentage** offered instead of the requested number.

It is **not** a non-answer when the manager answers briefly or imprecisely, or attaches the usual forward-looking-statement disclaimer and then answers.

**What the model needs.** Two things, and this drives the whole pipeline design:

1. **The question, not just the answer.** "About 20%" is a perfectly good answer to one question and a deflection from another. This is why Step 2 goes to the trouble of reconstructing question–answer *pairs* rather than just splitting the transcript into paragraphs.
2. **The coding rules themselves.** The boundary between a terse answer and a non-answer is genuinely fuzzy — de Kok notes it is "arguably somewhat subjective" even for human coders. Whatever line you want drawn has to be written into the prompt, or the model will draw its own.

**Do the task by hand first.** de Kok's advice, and it is not optional: code 20 or 30 pairs yourself before you write the prompt, and ask *why* you decided what you decided. Every rule you find yourself using is a rule the model needs to be told. You will do this formally in Step 6.

## Step 2. From a raw transcript to Q&A pairs

The unit of observation in de Kok (2025) is the **question–answer pair**, not the call: 1,152,505 pairs from 63,959 calls. Our sample is far smaller — 13 Tesla Q4 calls — but the plumbing is identical, and this step is pure Python: no LLM involved yet.

The transcripts are plain-text markdown with a consistent layout:

```
# Presentation
...
# Question-and-Answer Session

**Daniel Roeska**
*Bernstein*

Hey, good evening, everybody. ...

**Elon Musk**
*CEO*

Well, I mean, we're, I think, working on perfecting real-world AI ...
```

So every speaker turn is `**Name**`, an optional `*Title*`, and then the text. Let us look at the real thing first.

In [ ]:
CALL = CALL_DIR / "TSLA_2024Q4.txt"

raw = CALL.read_text(encoding="utf-8")
print(raw[:900])

### 2.1 Keep only the Q&A section

Non-answers can only occur in response to a question, so the prepared remarks in the presentation section are dropped. `str.partition()` splits a string on the first occurrence of a separator and returns `(before, separator, after)`.

In [ ]:
presentation, sep, qa = raw.partition("# Question-and-Answer Session")
assert sep, f"No Q&A section found in {CALL.name}"

print(f"presentation: {len(presentation):>6,} characters")
print(f"Q&A section : {len(qa):>6,} characters")

### 2.2 Split the Q&A section into speaker turns

One regular expression finds every speaker header; the text of a turn is everything up to the *next* header. Two details worth noting:

* `(?:\*(?P<title>[^*\n]*)\*\n)?` makes the title line **optional** — the operator has no title.
* Some transcript vendors prefix names with `Q - ` / `A - `; we strip that.

In [ ]:
TURN_RE = re.compile(r"^\*\*(?P<speaker>[^*\n]+)\*\*\n(?:\*(?P<title>[^*\n]*)\*\n)?", re.M)


def split_turns(section: str) -> list[dict]:
    """Split a Q&A section into a list of {speaker, title, text} turns."""
    turns = []
    headers = list(TURN_RE.finditer(section))
    for i, header in enumerate(headers):
        end = headers[i + 1].start() if i + 1 < len(headers) else len(section)
        turns.append({
            "speaker": re.sub(r"^[QA] - ", "", header.group("speaker")).strip(),
            "title": (header.group("title") or "").strip(),
            # collapse the hard line breaks inside a turn into single spaces
            "text": " ".join(section[header.end():end].split()),
        })
    return turns


turns = split_turns(qa)
print(f"{len(turns)} speaker turns")
pd.DataFrame(turns).head(6)

### 2.3 Who is asking and who is answering?

To pair questions with answers we need to know each speaker's role. The transcripts do not label this, so we infer it from the **title line**: management speakers get a job title (*CEO*, *CFO*, *VP, Vehicle Engineering*, or simply *Tesla, Inc.*), while analysts and journalists get their employer (*Morgan Stanley*, *Bernstein*, *New Street Research*).

Investor-relations staff are treated as a separate role, because in Tesla's recent calls the head of IR **reads out retail-investor questions submitted through say.com** before handing over to the sell-side analysts. Ignore those turns and you throw away half of the questions asked after 2019.

This kind of heuristic is transcript-specific and is exactly the sort of thing you must **look at**, never assume — so we print the classification and check it.

In [ ]:
MGMT_TITLE = re.compile(
    r"CEO|CFO|CTO|Chief|President|Officer|Chairman|Founder|Technoking|"
    r"Architect|Executive|Engineering|Director|VP|Tesla",
    re.I,
)
IR_TITLE = re.compile(r"Investor Relations|\bIR\b", re.I)


def role(turn: dict) -> str:
    """Classify a speaker turn as operator / ir / management / analyst."""
    if turn["speaker"].lower().startswith("operator"):
        return "operator"
    if IR_TITLE.search(turn["title"]):
        return "ir"
    if MGMT_TITLE.search(turn["title"]) or "Company Representative" in turn["speaker"]:
        return "management"
    return "analyst"


turns_df = pd.DataFrame(turns).assign(role=lambda d: d.apply(role, axis=1))
turns_df.groupby(["role", "speaker", "title"]).size().rename("turns").reset_index()

### 2.4 Build the Q&A pairs

The pairing rule:

* a **question** is a turn by an analyst (or journalist), or an IR turn that contains a question mark;
* the **answer** is every management turn that follows it, concatenated — if the CFO adds to the CEO's reply, that is still one answer to one question (de Kok, endnote 7).

We then apply his length filters, which drop the "Q: Thank you – A: Thanks!" pairs that would otherwise inflate the accuracy statistics: question ≥ 30 characters, answer ≥ 10 characters, and the two together ≥ 75 characters.

In [ ]:
MIN_Q, MIN_A, MIN_BOTH = 30, 10, 75  # de Kok (2025), endnote 7


def qa_pairs(turns: list[dict]) -> list[dict]:
    """Pair each question turn with the management turns that answer it."""
    pairs, i = [], 0
    while i < len(turns):
        turn = turns[i]
        turn_role = role(turn)
        asks_question = turn_role == "analyst" or (turn_role == "ir" and "?" in turn["text"])
        if not asks_question:
            i += 1
            continue

        answer_parts, j = [], i + 1
        while j < len(turns) and role(turns[j]) == "management":
            answer_parts.append(turns[j]["text"])
            j += 1
        answer = " ".join(answer_parts)

        if (len(turn["text"]) >= MIN_Q and len(answer) >= MIN_A
                and len(turn["text"]) + len(answer) >= MIN_BOTH):
            pairs.append({
                "asker": turn["speaker"],
                "asker_role": turn_role,
                "question": turn["text"],
                "answer": answer,
            })
        i = max(j, i + 1)
    return pairs


def load_call(path: Path) -> pd.DataFrame:
    """Read one transcript file and return its Q&A pairs as a DataFrame."""
    _, _, section = path.read_text(encoding="utf-8").partition("# Question-and-Answer Session")
    df = pd.DataFrame(qa_pairs(split_turns(section)))
    df.insert(0, "call", path.stem)
    df.insert(1, "pair_id", [f"{path.stem}_{k:03d}" for k in range(len(df))])
    return df


pairs = load_call(CALL)
print(f"{len(pairs)} Q&A pairs in {CALL.stem}")
pairs.head()

In [ ]:
# Always eyeball what you are about to send to the model.
example = pairs.iloc[0]
print("Q:", example["question"][:600], "\n")
print("A:", example["answer"][:600])

## Step 3. Decide on the approach and model

Three ways to instruct a generative LLM (de Kok, §3.2):

| Approach | What it is | Trade-off |
| --- | --- | --- |
| **Zero shot** | instructions + data, no examples | easiest; least control |
| **Few shot** | instructions + a handful of labelled examples in every prompt | better on nuanced tasks; more tokens per observation |
| **Fine-tuning** | retrain the model weights on a labelled training set | most control; needs a training set and much more work |

The recommendation is to start at the top of that table and only move down if construct validity is unsatisfactory, and to pick the **smallest model that performs well enough** — not the best model. For 356 Q&A pairs, a zero-shot call to a current model costs cents; de Kok's full sample of 1.2 million pairs cost about \$1,300 with a funnel of cheap filters in front of the LLM, and would have cost roughly \$20,500 with zero-shot GPT-4 at September 2023 prices. Cost only becomes the binding constraint at scale, but at scale it binds hard.

We use `gpt-5.6` zero-shot, which corresponds to column (3) of his Table 1.

## Step 4. Develop the prompt

Four ideas from the paper are built into the prompt below (§3.3 and Appendix C):

1. **Give the model the coding rules**, including what is *not* a non-answer. Boilerplate forward-looking disclaimers followed by a real answer are the single biggest source of false positives.
2. **Make the output machine-readable.** We use a Pydantic schema with `client.responses.parse()`, so the completion comes back as a validated object instead of prose we have to regex.
3. **Ask for the assessment *before* the label.** LLMs generate left to right, so their own reasoning becomes part of the context for the token that follows — the classic chain-of-thought effect. Field order in the schema is therefore a design decision, not cosmetics.
4. **State the expected distribution.** Each call is independent, so the model has no idea that non-answers are rare and will happily flag a third of the sample. de Kok reports that deleting his "these sentences are rare, in 65% of the cases..." sentence *significantly* degrades performance.

In [ ]:
INSTRUCTIONS = """
You are a research assistant coding earnings conference call transcripts for an
academic accounting study. Apply the coding rules exactly as written and base your
coding only on the text you are given.
"""

PROMPT = """
Below is one question-and-answer pair from the Q&A section of an earnings conference call.

Analyst question:
{question}

Manager response:
{answer}

A NON-ANSWER is a response in which the manager indicates an inability or an unwillingness
to provide (part of) the information the question asked for. It includes:
- declining for proprietary, competitive, policy or legal reasons ("we don't disclose that",
  "we don't give guidance on that");
- saying they do not know, do not have the information, or that it is too early to tell;
- deflecting the question with a purely qualitative statement or a broad range instead of
  the specific information that was requested.

It is NOT a non-answer when the manager:
- provides the requested information, even briefly, roughly or imprecisely;
- adds a boilerplate disclaimer (e.g. about forward-looking statements) and then answers;
- discusses general uncertainty about the future while still answering the question asked.

Your task:
1. Assess the response in the context of the question that was asked. If it contains a
   non-answer, quote the sentence(s) from the response that show it.
2. Then classify the pair: 1 if the response contains a non-answer, 0 otherwise.

Important: non-answers are relatively rare - for roughly 85% of pairs the correct
classification is 0. It is fine, and expected, to return 0 most of the time.
"""

In [ ]:
class NonAnswerCoding(BaseModel):
    """Schema for the completion. Field order = the order the model generates them in."""

    assessment: str = Field(
        description="Two or three sentences assessing the response in the context of the question."
    )
    evidence: str = Field(
        description="Verbatim sentence(s) from the response showing the non-answer; "
                    "an empty string if there is none."
    )
    nonanswer: int = Field(
        description="1 if the response contains a non-answer, 0 otherwise."
    )

In [ ]:
def classify_pair(question: str, answer: str, model: str = MODEL) -> tuple[NonAnswerCoding, dict]:
    """Classify one Q&A pair. Returns the parsed coding and a raw record for the log."""
    prompt = PROMPT.format(question=question, answer=answer)
    response = client.responses.parse(
        model=model,
        instructions=INSTRUCTIONS,
        input=prompt,
        text_format=NonAnswerCoding,
    )
    coding = response.output_parsed
    record = {
        "model": model,
        "prompt": prompt,
        "completion": coding.model_dump(),
        "usage": response.usage.model_dump() if response.usage else {},
    }
    return coding, record

In [ ]:
# One pair, to see what comes back before spending money on the rest.
coding, record = classify_pair(example["question"], example["answer"])
coding

In [ ]:
# Tokens drive both the cost and the speed - check them before scaling up.
print(f"input tokens: {record['usage'].get('input_tokens')}, "
      f"output tokens: {record['usage'].get('output_tokens')}")

## Step 5. Run the classification over the call

Two things the loop below does that a throwaway script would not, both from §5.2 of the paper:

* **It logs the raw prompt and the raw completion for every observation** to a JSONL file. Third-party providers retire and silently change models; generations are not perfectly reproducible even at the same model version. Your prompts and completions are the primary data of the study — treat them the way you treat a raw WRDS download, and never regenerate them casually.
* **It skips pairs that are already in the log.** A crashed run resumes where it stopped, and re-running the cell costs nothing. (Delete the log file to force a fresh run.)

`ThreadPoolExecutor` sends a few requests at a time. Requests spend nearly all their time waiting on the network, so a handful of threads shortens a 356-call job from tens of minutes to a few — keep `max_workers` modest to stay clear of the API rate limits.

In [ ]:
RAW_LOG = OUT_DIR / "nonanswer_raw.jsonl"


def load_raw(log: Path = RAW_LOG) -> pd.DataFrame:
    """Read the raw log back into a DataFrame, one row per classified pair."""
    if not log.exists():
        return pd.DataFrame(columns=["pair_id", "call", "assessment", "evidence", "nonanswer"])
    records = [json.loads(line) for line in log.read_text().splitlines() if line.strip()]
    return pd.DataFrame([
        {"pair_id": r["pair_id"], "call": r["call"], **r["completion"]} for r in records
    ]).drop_duplicates(subset="pair_id", keep="last")


def classify_frame(df: pd.DataFrame, model: str = MODEL,
                   log: Path = RAW_LOG, max_workers: int = 4) -> None:
    """Classify every pair in `df` that is not already in the log, appending as we go."""
    done = set(load_raw(log)["pair_id"])
    todo = df[~df["pair_id"].isin(done)]
    print(f"{len(todo)} pairs to classify, {len(df) - len(todo)} already in the log")
    if todo.empty:
        return

    with ThreadPoolExecutor(max_workers=max_workers) as pool, log.open("a") as fh:
        futures = {
            pool.submit(classify_pair, row.question, row.answer, model): row
            for row in todo.itertuples()
        }
        for n, future in enumerate(as_completed(futures), start=1):
            row = futures[future]
            try:
                _, record = future.result()
            except Exception as exc:  # keep the completed work, report the failure
                print(f"\n{row.pair_id} failed: {exc}")
                continue
            fh.write(json.dumps({"pair_id": row.pair_id, "call": row.call, **record}) + "\n")
            fh.flush()
            print(f"\r{n}/{len(futures)} classified", end="")
    print()

In [ ]:
classify_frame(pairs)

In [ ]:
results = pairs.merge(load_raw().drop(columns="call"), on="pair_id")

print(f"{results['nonanswer'].mean():.1%} of the {len(results)} Q&A pairs in "
      f"{CALL.stem} contain a non-answer")
print("(de Kok (2025) finds 13.9% across 1.15 million pairs)")
results[["asker", "asker_role", "nonanswer", "evidence"]].head(10)

In [ ]:
# Read the model's reasoning on the pairs it flagged - this is where you catch a bad prompt.
for row in results[results["nonanswer"] == 1].head(3).itertuples():
    print(f"--- {row.pair_id} ({row.asker}) ---")
    print("Q:", row.question[:300])
    print("A:", row.answer[:300])
    print("EVIDENCE  :", row.evidence)
    print("ASSESSMENT:", row.assessment, "\n")

## Step 6. Evaluate the construct validity

> *"Outputs generated by GLLMs tend to look great at first inspection but can be less accurate upon closer inspection. Because of human confirmation bias, it is easy to overestimate the quality of a GLLM completion."* — de Kok (2025, §3.4)

The output above *looks* convincing. That is not evidence. A zero-shot approach needs no training data, but it still needs an **evaluation set**, and the same performance statistics you would report for any supervised classifier.

We do two things here:

1. Compare the LLM against a **keyword baseline** — a crude stand-in for the rule-based Gow et al. (2021) measure — and look at where they disagree.
2. Build a **hand-coded evaluation sample** and score the LLM against it.

### 6.1 A keyword baseline

Not a replication of Gow et al. (2021), just the same idea in miniature: flag a response if it contains one of a list of non-answer phrases. Rules like this are cheap and fast, which is why they dominated the literature — and they are why detecting non-answers was still an open problem.

In [ ]:
NONANSWER_PHRASES = [
    "don't disclose", "do not disclose", "not disclosing", "won't disclose",
    "don't provide", "do not provide", "not going to provide", "no guidance",
    "don't give guidance", "not providing guidance", "don't break out", "do not break out",
    "no comment", "can't comment", "cannot comment", "not going to comment",
    "can't give", "cannot give", "can't share", "can't get into", "won't get into",
    "not going to get into", "too early to", "don't know", "do not know",
    "not sure", "get back to you", "stay tuned",
]


def keyword_nonanswer(answer: str) -> int:
    """Baseline: flag the response if it contains any known non-answer phrase."""
    low = answer.lower()
    return int(any(phrase in low for phrase in NONANSWER_PHRASES))


results["keyword"] = results["answer"].apply(keyword_nonanswer)

print(f"GPT method     : {results['nonanswer'].mean():.1%} non-answers")
print(f"Keyword method : {results['keyword'].mean():.1%} non-answers")
pd.crosstab(results["keyword"], results["nonanswer"],
            rownames=["keyword"], colnames=["gpt"])

In [ ]:
# The aggregate rates can agree while the classifications disagree - de Kok's Figure 2 makes
# exactly this point. Read the disagreements: which method is right?
disagree = results[results["keyword"] != results["nonanswer"]]
for row in disagree.head(3).itertuples():
    print(f"--- {row.pair_id}: keyword={row.keyword}, gpt={row.nonanswer} ---")
    print("Q:", row.question[:250])
    print("A:", row.answer[:400])
    print("ASSESSMENT:", row.assessment, "\n")

### 6.2 Hand-code an evaluation sample

The cell below writes a coding template with **no model output in it** — seeing the label first is the fastest way to talk yourself into agreeing with it.

**Your turn:**

1. Open `data/nonanswer/manual_coding_template.csv`.
2. Fill the `manual` column with 1 (non-answer) or 0 (answer), using the rules from Step 1.
3. Save it as `data/nonanswer/manual_coding.csv` and run the scoring cell.

Note what happens while you code: the hard cases are not the model's fault. They are cases where *you* are unsure, and where a second coder would plausibly disagree. That ambiguity is a property of the construct, and it puts a ceiling on any measure of it — human or machine.

For a paper, de Kok recommends an evaluation sample of a hundred to a couple of thousand pairs, drawn to reflect the diversity of the data. Twenty is for the classroom.

In [ ]:
TEMPLATE_FILE = OUT_DIR / "manual_coding_template.csv"
CODED_FILE = OUT_DIR / "manual_coding.csv"

if not TEMPLATE_FILE.exists():
    template = (results[["pair_id", "question", "answer"]]
                .sample(min(20, len(results)), random_state=42)
                .assign(manual=""))          # deliberately no model output here
    template.to_csv(TEMPLATE_FILE, index=False)

print(f"Template: {TEMPLATE_FILE}")
print(f"Save your codings as: {CODED_FILE}")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

if CODED_FILE.exists():
    coded = pd.read_csv(CODED_FILE).dropna(subset=["manual"])
    scored = coded[["pair_id", "manual"]].merge(results, on="pair_id")

    for name, pred in [("GPT method", scored["nonanswer"]), ("Keyword baseline", scored["keyword"])]:
        print(f"===== {name} (n = {len(scored)}) =====")
        print(confusion_matrix(scored["manual"], pred))
        print(classification_report(scored["manual"], pred,
                                    target_names=["answer", "non-answer"], zero_division=0))
else:
    print(f"No hand-coded file yet - fill in {TEMPLATE_FILE.name} and save it as {CODED_FILE.name}.")

When you read that report, read the **non-answer row**, not the accuracy. Non-answers are roughly 14% of pairs, so a model that labels everything 0 already scores 86% accuracy — de Kok's endnote 10 makes this point about the Gow et al. baseline, whose 86% accuracy conceals a 0.43 recall. Precision, recall and F1 **on the minority class** are what tell you whether the measure works.

## Step 7. Scale up: inference over all 13 calls

Once construct validity is satisfactory, the same code runs over the full sample — what de Kok calls model inference. Ours is 13 calls and roughly 356 pairs, so a few minutes and a few cents. Before you launch a job like this on real data, do what §3.2 recommends: measure the tokens on a handful of prompts and extrapolate to the full sample **before** you press run.

The log means this cell can be interrupted and re-run safely.

In [ ]:
all_pairs = pd.concat([load_call(f) for f in CALL_FILES], ignore_index=True)

# --- In class, start small! Uncomment to cap the number of pairs per call. ---
# all_pairs = all_pairs.groupby("call", group_keys=False).head(5)

print(f"{len(all_pairs)} Q&A pairs across {all_pairs['call'].nunique()} calls")
all_pairs.groupby("call").size().rename("pairs")

In [ ]:
classify_frame(all_pairs, max_workers=8)

In [ ]:
full = all_pairs.merge(load_raw().drop(columns="call"), on="pair_id")
full["keyword"] = full["answer"].apply(keyword_nonanswer)
full["year"] = full["call"].str[5:9].astype(int)

by_year = full.groupby("year").agg(
    pairs=("nonanswer", "size"),
    gpt=("nonanswer", "mean"),
    keyword=("keyword", "mean"),
)
by_year

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(by_year.index, by_year["gpt"], "o-", label="GPT method")
plt.plot(by_year.index, by_year["keyword"], "s--", label="Keyword baseline")
plt.axhline(0.139, color="grey", linestyle=":", linewidth=1,
            label="de Kok (2025), full sample: 13.9%")
plt.xlabel("Fiscal year (Q4 call)")
plt.ylabel("Share of Q&A pairs with a non-answer")
plt.title("Non-answers in Tesla's Q4 earnings calls")
plt.legend()
plt.tight_layout()
plt.show()

Treat that figure as a demonstration of the pipeline, not as a finding. Each point rests on 14–46 Q&A pairs from a single firm, so the year-to-year movement is mostly sampling noise; de Kok's own time-series evidence (his Figure 4, where non-answers rise during COVID-19) rests on 1.15 million pairs and a four-quarter moving average. Scaling to a real research sample means a transcript provider (he uses Finnhub.io) and a cost model — the code above does not change.

## Step 8. Your turn

**1. Non-answer dimensions.** Section 4.2 of the paper classifies *why* the manager did not answer, *how*, and *what was asked*, all in the same call. The cell below is a starting schema — adapt `PROMPT` and rerun on the flagged pairs, then check the results against a few you code yourself.

**2. Test the prompt's robustness.** Delete the "non-answers are relatively rare" sentence, or the list of what is *not* a non-answer, and rerun on the same call. How much does the non-answer rate move? de Kok found this single sentence materially changed performance — an uncomfortable result worth seeing for yourself, because it means the measure depends on wording you chose.

**3. Change the model.** Rerun the call with a smaller/cheaper model (`classify_frame(pairs, model=..., log=OUT_DIR / "other_model.jsonl")`) and compare against your hand-coded sample. If the cheap model is as good on *your* task, use it — the recommendation is the smallest model that performs well enough, not the best one available.

**4. Fix a real limitation of Step 2.** IR turns that contain a question mark are treated as questions, so an IR reply that happens to end in a rhetorical question becomes a spurious "question", and a retail question read out without a question mark is missed. Quantify how often this happens, and write a better rule.

**5. Take it to your own data.** Any textual construct with a clear coding rule works the same way: risk-factor changes, disclosure tone, humour in earnings calls, ESG target extraction. The pipeline is identical — pairs in, schema out, evaluate against hand coding.

In [ ]:
from typing import Literal


class NonAnswerDimensions(BaseModel):
    """de Kok (2025), Appendix A: the dimensions of a non-answer."""

    assessment: str
    nonanswer: int
    justification: Literal["cannot give", "do not know", "not applicable"]
    nonanswer_type: Literal["complete refusal", "qualitative", "range or percentage", "not applicable"]
    question_type: Literal["breakdown", "forward-looking", "related party", "R&D or regulation", "other"]
    sentiment_spin: Literal["optimistic", "neutral or pessimistic", "not applicable"]